# **Project Name** - Amazon Prime Video IMDb Rating Analysis

##### **Project Type**    - EDA/Regression/Classification/Unsupervised
**Selected project type** - Regression and EDA
##### **Contribution**    - Individual/Team
**Selected contribution** - Individual
##### **Name -** Vishesh Rajak

# **Project Summary -**

This project builds a regression workflow to estimate a title's public IMDb score from metadata that could be known before launch or acquisition. The supplied brief describes a US Amazon Prime Video catalog snapshot. The titles file contains 9,871 rows and 15 fields, while the credits file contains 124,235 actor and director credit rows. Metadata includes format, release year, runtime, certification, description, genre and production-country lists, IMDb ratings and votes, and TMDB measures. Credits support exploratory talent analysis, but names are excluded from prediction because they are high-cardinality and unavailable consistently for an uncast project.

The target is IMDb score on a 0–10 scale. Three repeated title IDs and 56 exact duplicate credit rows are removed. The target is never imputed; titles without an observed score are excluded from supervised training. Numeric predictors are imputed within model pipelines, missing categories become Unknown, and multi-label fields are parsed safely. Description text is cleaned with deterministic rules and represented with TF-IDF inside the training pipeline. Structured predictors include release year, type, runtime, show season count, certification, genre and country text, and synopsis length.

The model excludes IMDb votes, IMDb score, TMDB score, TMDB popularity, IDs, and title names from features. These fields can be missing at acquisition time, encode post-release reception, or let a model memorize records. This feature boundary makes the exercise closer to a pre-release quality estimate. An 80/20 stratified holdout preserves the movie/show mix. The workflow compares Ridge regression with Random Forest and Extra Trees. Each model gets a baseline fit, then a compact parameter search with three-fold cross-validation selects settings using RMSE. The final model is selected by cross-validation RMSE within training data and evaluated on the holdout.

The notebook reports mean absolute error, root mean squared error, and R-squared. MAE describes typical error in IMDb points. RMSE penalizes large misses more strongly. R-squared compares explained score variation with predicting a constant mean. Score charts compare baseline and tuned models, and a feature chart explains the selected estimator using absolute Ridge coefficients or tree importances. Preprocessing is fit inside each pipeline, so validation folds do not learn imputation values, categories, or text vocabulary from their validation rows.

The model may help analysts prioritize titles for human review when they have descriptive metadata. It cannot forecast Prime subscribers, watch time, licensing cost, retention, or profitability. Public ratings are self-selected audience responses and are unavailable for 1,021 titles. Lower prediction error does not prove that metadata causes ratings. Before production use, validate on later release years, report performance by format and market, monitor rating coverage and drift, and join first-party engagement and cost data. The source ends in 2022 and does not represent a current live catalog. Treat each prediction as an uncertain estimate and inspect error distributions before use.

Because these files contain no availability dates or recommendation exposure, the model cannot separate awareness from preference. Training excludes unrated titles, so estimates apply to the rated subset and may be less reliable for titles with little exposure. Review predicted scores alongside uncertainty and domain knowledge.

# **GitHub Link -**

```markdown
**GitHub Link:** https://github.com/VisheshRajak/Amazon_Prime_Video_Project_Ds-Aiml

# **Problem Statement**


The content team needs a consistent way to screen a large catalog while balancing audience quality with acquisition risk. Public scores are unevenly available, formats differ, and metadata is incomplete. A model trained on post-release votes, popularity, or external ratings would leak information unavailable when a title is considered. This project predicts IMDb score using release year, type, runtime, seasons, certification, genre, country, and description features only. IMDb score is a proxy for public reception, not a measure of Prime viewing, retention, cost, or return on investment.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries

import ast
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
import ast
import warnings
import re
import unicodedata
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
print('Setup completed successfully!')

In [ ]:
# Automated full pipeline execution script
titles_path = None
credits_path = None

# Look for dataset paths
for p in [Path('/content/titles.csv'), Path('titles.csv')]:
    if p.is_file():
        titles_path = p
        break

for p in [Path('/content/credits.csv'), Path('credits.csv')]:
    if p.is_file():
        credits_path = p
        break

if titles_path is None or credits_path is None:
    print('Warning: Please ensure titles.csv and credits.csv are placed in your current directory.')
else:
    titles_raw_df = pd.read_csv(titles_path)
    credits_raw_df = pd.read_csv(credits_path)
    print(f'Datasets loaded: Titles shape {titles_raw_df.shape}, Credits shape {credits_raw_df.shape}')

### Dataset Loading

In [ ]:
# Load Dataset
titles_raw_df = pd.read_csv('/content/titles.csv')
credits_raw_df = pd.read_csv('/content/credits.csv')
print('Titles shape:', titles_raw_df.shape)
print('Credits shape:', credits_raw_df.shape)

### Dataset First View

In [ ]:
# Dataset First Look
display(titles_raw_df.head())
display(credits_raw_df.head())

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print('Titles shape (rows, columns):', titles_raw_df.shape)
print('Credits shape (rows, columns):', credits_raw_df.shape)

### Dataset Information

In [ ]:
# Dataset Info
titles_raw_df.info()
print('\nCredits data types:')
display(credits_raw_df.dtypes.to_frame('dtype'))

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print('Duplicate title IDs:', titles_raw_df['id'].duplicated().sum())
print('Duplicate credit rows:', credits_raw_df.duplicated().sum())

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
display(titles_raw_df.isna().sum().sort_values(ascending=False).to_frame('missing_rows'))
print('Credit missing values:')
display(credits_raw_df.isna().sum().to_frame('missing_rows'))

In [ ]:
# Visualizing the missing values

missing_pct=titles_raw_df.isna().mean().mul(100).sort_values(ascending=False)
fig,ax=plt.subplots(figsize=(10,4)); missing_pct[missing_pct.gt(0)].plot(kind='bar',ax=ax,color='#F58518')
ax.set(title='Missing-value share in original title data',xlabel='Field',ylabel='Rows missing (%)')
ax.tick_params(axis='x',rotation=45); fig.tight_layout(); plt.show()

### What did you know about your dataset?

The cleaned title table has 9,868 unique title IDs after removing three repeated IDs. It covers 8,514 movies and 1,357 shows, with release years from 1912 through 2022. There are 8,850 non-missing IMDb scores. Movie rows dominate. Show season counts are structurally absent for movies, so season analysis is limited to shows. The credits table has 124,179 distinct rows after removing 56 exact duplicates and includes actor and director roles. Genre and production-country lists can contain multiple values per title, so exploded counts overlap. The project brief describes US catalog availability. The files do not provide user viewing, costs, licensing terms, retention, or time-stamped availability history.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
print("Titles Columns:", list(titles_raw_df.columns))
print("Credits Columns:", list(credits_raw_df.columns))

In [ ]:
# Dataset Describe
if 'titles_df' in globals():
    display(titles_df.describe().T.round(2))
else:
    print("Using raw titles dataset for description:")
    display(titles_raw_df.describe().T.round(2))

### Variables Description

| Field | Meaning | Analysis use |
|---|---|---|
| id, title, imdb_id | Title identifiers and name | Join and audit; excluded from prediction |
| type | MOVIE or SHOW | Format comparison and categorical predictor |
| description | Short synopsis | Text exploration and TF-IDF feature |
| release_year | Release year | Trend analysis and numeric predictor |
| age_certification | Audience certification label | Coverage audit and categorical predictor |
| runtime | Runtime in minutes | Numeric comparison; show runtime is episode-level |
| genres | Multi-label genre list | Genre analysis and text-vectorized predictor |
| production_countries | Multi-label country codes | Market mix and text-vectorized predictor |
| seasons | Number of seasons for shows | Show-only analysis; structurally absent for movies |
| imdb_score, imdb_votes | Public score and vote count | Outcome and description; votes excluded from pre-release model |
| tmdb_popularity, tmdb_score | Public TMDB measures | Descriptive analysis; excluded from pre-release model |
| person_id, name, character, role | Credited person, character and role | Talent summary after joining on title id |

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable safely handling list types
target_df = titles_df if 'titles_df' in globals() else titles_raw_df
unique_counts = pd.DataFrame({
    'column': target_df.columns,
    'unique_values': [target_df[c].map(lambda x: str(x) if isinstance(x, list) else x).nunique(dropna=True) for c in target_df.columns]
})
display(unique_counts)

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
import ast
import pandas as pd

def safe_parse_list(value):
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed=ast.literal_eval(value) if isinstance(value,str) else value
        return parsed if isinstance(parsed,list) else []
    except (ValueError,SyntaxError,TypeError):
        return []

titles_df=titles_raw_df.drop_duplicates(subset='id',keep='first').copy()
credits_df=credits_raw_df.drop_duplicates().copy()
for col in ['runtime','release_year','seasons','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']:
    titles_df[col]=pd.to_numeric(titles_df[col],errors='coerce')
titles_df['genres_list']=titles_df['genres'].map(safe_parse_list)
titles_df['production_countries_list']=titles_df['production_countries'].map(safe_parse_list)
titles_df['age_certification']=titles_df['age_certification'].fillna('Unknown').replace('','Unknown')
titles_df['description']=titles_df['description'].fillna('').astype(str)
show_median_seasons=titles_df.loc[titles_df['type'].eq('SHOW'),'seasons'].median()
titles_df['seasons_for_model']=titles_df['seasons'].fillna(0)
titles_df.loc[titles_df['type'].eq('SHOW'),'seasons_for_model']=titles_df.loc[titles_df['type'].eq('SHOW'),'seasons'].fillna(show_median_seasons)
titles_df['decade']=(titles_df['release_year']//10*10).astype('Int64')
print('Titles after deduplication:',titles_df.shape)
print('Credit rows after exact deduplication:',credits_df.shape)
print('Remaining missing IMDb scores:',titles_df['imdb_score'].isna().sum())

### What all manipulations have you done and insights you found?

Removed three duplicate title IDs and 56 exact duplicate credit rows. Parsed genre and production-country strings with a safe list parser. Converted numeric fields and added decade and list-count features for charts. Labeled missing certification as Unknown and filled missing descriptions with empty text only for text processing. Movie season counts are structurally not applicable; show seasons are summarized among shows only. Rating charts exclude missing IMDb scores. Talent ratings are joined by title ID and restricted to people with at least ten credited titles to reduce single-title noise.

For supervised learning, rows without the IMDb target are excluded rather than assigned a fabricated score. Predictors are imputed inside model pipelines. IMDb score, vote counts, TMDB measures, IDs, title names, and credited names are excluded to reduce target leakage and memorization.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code

def plot_catalog_chart(number):
    """Draw a reproducible chart from the cleaned title and credit tables."""
    if number == 1:
        counts=titles_df['type'].value_counts().reindex(['MOVIE','SHOW']).dropna()
        fig,ax=plt.subplots(figsize=(7,4)); bars=ax.bar(counts.index,counts.values,color=['#4C78A8','#F58518'])
        ax.bar_label(bars,fmt='{:,.0f}'); ax.set(title='Catalog titles by format',xlabel='Format',ylabel='Titles')
    elif number == 2:
        fig,ax=plt.subplots(figsize=(10,4)); sns.histplot(data=titles_df,x='release_year',hue='type',multiple='stack',binwidth=5,ax=ax)
        ax.set(title='Release-year distribution in five-year bins',xlabel='Release year',ylabel='Titles')
    elif number == 3:
        fig,ax=plt.subplots(figsize=(8,4)); sns.boxplot(data=titles_df,x='type',y='runtime',showfliers=False,ax=ax)
        ax.set(title='Runtime differs by format',xlabel='Format',ylabel='Runtime (minutes)')
    elif number == 4:
        d=titles_df.dropna(subset=['imdb_score']); fig,ax=plt.subplots(figsize=(8,4))
        sns.histplot(data=d,x='imdb_score',bins=25,kde=True,color='#4C78A8',ax=ax)
        ax.set(title='IMDb score distribution',xlabel='IMDb score (0–10)',ylabel='Rated titles')
    elif number == 5:
        d=titles_df.loc[titles_df['type'].eq('SHOW')].dropna(subset=['seasons'])
        fig,ax=plt.subplots(figsize=(8,4)); sns.histplot(data=d,x='seasons',discrete=True,color='#F58518',ax=ax)
        ax.set(title='Season counts for shows',xlabel='Number of seasons',ylabel='Shows')
    elif number == 6:
        d=titles_df.dropna(subset=['imdb_score']); fig,ax=plt.subplots(figsize=(8,4))
        sns.boxplot(data=d,x='type',y='imdb_score',showfliers=False,ax=ax)
        ax.set(title='IMDb scores by format',xlabel='Format',ylabel='IMDb score')
    elif number == 7:
        d=titles_df.groupby(['release_year','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,4)); sns.lineplot(data=d,x='release_year',y='titles',hue='type',estimator=None,ax=ax)
        ax.set(title='Annual catalog volume by format',xlabel='Release year',ylabel='Titles in snapshot')
    elif number == 8:
        d=titles_df.assign(decade=titles_df['release_year']//10*10).groupby(['decade','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,4)); sns.barplot(data=d,x='decade',y='titles',hue='type',ax=ax)
        ax.tick_params(axis='x',rotation=45); ax.set(title='Catalog volume by decade and format',xlabel='Release decade',ylabel='Titles')
    elif number == 9:
        d=titles_df.dropna(subset=['imdb_score']).assign(decade=lambda x:x['release_year']//10*10).groupby('decade',as_index=False)['imdb_score'].mean()
        fig,ax=plt.subplots(figsize=(9,4)); sns.lineplot(data=d,x='decade',y='imdb_score',marker='o',ax=ax)
        ax.set(title='Mean IMDb score by release decade',xlabel='Release decade',ylabel='Mean IMDb score',ylim=(0,10))
    elif number == 10:
        d=titles_df.dropna(subset=['runtime','imdb_score']); d=d.sample(min(3000,len(d)),random_state=42)
        fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='runtime',y='imdb_score',hue='type',alpha=.35,s=22,ax=ax)
        ax.set(title='Runtime and IMDb score',xlabel='Runtime (minutes)',ylabel='IMDb score')
    elif number == 11:
        d=titles_df.dropna(subset=['imdb_votes','imdb_score']).sample(frac=1,random_state=42)
        d=d.head(min(3000,len(d))); fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='imdb_votes',y='imdb_score',alpha=.35,s=22,ax=ax)
        ax.set_xscale('log'); ax.set(title='IMDb votes and score',xlabel='IMDb votes (log scale)',ylabel='IMDb score')
    elif number == 12:
        d=titles_df.dropna(subset=['tmdb_popularity','imdb_score']).sample(frac=1,random_state=42)
        d=d.head(min(3000,len(d))); fig,ax=plt.subplots(figsize=(8,5)); sns.scatterplot(data=d,x='tmdb_popularity',y='imdb_score',alpha=.35,s=22,ax=ax)
        ax.set_xscale('log'); ax.set(title='TMDB popularity and IMDb score',xlabel='TMDB popularity (log scale)',ylabel='IMDb score')
    elif number == 13:
        d=titles_df[['genres_list']].explode('genres_list').dropna(subset=['genres_list'])
        s=d['genres_list'].value_counts().head(12).sort_values()
        fig,ax=plt.subplots(figsize=(9,5)); s.plot(kind='barh',ax=ax,color='#4C78A8')
        ax.set(title='Most common genres',xlabel='Titles tagged with genre',ylabel='Genre')
    elif number == 14:
        cols=['release_year','runtime','seasons','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']
        corr=titles_df[cols].corr(numeric_only=True); fig,ax=plt.subplots(figsize=(9,7))
        sns.heatmap(corr,annot=True,fmt='.2f',cmap='vlag',center=0,vmin=-1,vmax=1,ax=ax)
        ax.set_title('Correlations among numeric title fields')
    elif number == 15:
        cols=['release_year','runtime','imdb_score','tmdb_score','type']; d=titles_df[cols].dropna()
        d=d.sample(min(700,len(d)),random_state=42)
        sns.pairplot(d,vars=cols[:-1],hue='type',corner=True,diag_kind='hist',plot_kws={'alpha':.35,'s':18})
        plt.suptitle('Pairwise patterns across a deterministic sample',y=1.02); plt.show(); return
    elif number == 16:
        d=titles_df[['production_countries_list']].explode('production_countries_list').dropna(subset=['production_countries_list'])
        s=d['production_countries_list'].value_counts().head(12).sort_values()
        fig,ax=plt.subplots(figsize=(9,5)); s.plot(kind='barh',ax=ax,color='#72B7B2')
        ax.set(title='Most represented production countries',xlabel='Titles tagged with country',ylabel='Country code')
    elif number == 17:
        d=titles_df[['production_countries_list','imdb_score']].explode('production_countries_list').dropna()
        s=d.groupby('production_countries_list')['imdb_score'].agg(['mean','count']).query('count >= 100').sort_values('mean').tail(12)
        fig,ax=plt.subplots(figsize=(9,5)); ax.barh(s.index,s['mean'],color='#54A24B')
        ax.set(title='Mean IMDb score by country (100+ rated titles)',xlabel='Mean IMDb score',ylabel='Country code',xlim=(0,10))
    elif number == 18:
        d=titles_df[['type','genres_list','imdb_score']].explode('genres_list').dropna()
        top=d['genres_list'].value_counts().head(10).index
        means=d[d['genres_list'].isin(top)].pivot_table(index='genres_list',columns='type',values='imdb_score',aggfunc='mean')
        fig,ax=plt.subplots(figsize=(8,6)); sns.heatmap(means,annot=True,fmt='.2f',cmap='YlGnBu',vmin=0,vmax=10,ax=ax)
        ax.set(title='Mean IMDb score by genre and format',xlabel='Format',ylabel='Genre')
    elif number == 19:
        d=titles_df.assign(age_certification=titles_df['age_certification'].fillna('Unknown')).groupby(['age_certification','type']).size().rename('titles').reset_index()
        fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=d,x='age_certification',y='titles',hue='type',ax=ax)
        ax.tick_params(axis='x',rotation=45); ax.set(title='Certification labels by format',xlabel='Certification',ylabel='Titles')
    elif number == 20:
        d=credits_df.merge(titles_df[['id','imdb_score']],on='id',how='inner').dropna(subset=['imdb_score'])
        d=d.groupby(['role','name']).agg(title_count=('id','nunique'),mean_score=('imdb_score','mean')).reset_index()
        fig,axes=plt.subplots(1,2,figsize=(14,6))
        for ax,role in zip(axes,['ACTOR','DIRECTOR']):
            top=d.query('role == @role and title_count >= 10').nlargest(10,'mean_score').sort_values('mean_score')
            ax.barh(top['name'],top['mean_score'],color='#E45756')
            ax.set(title=f'Top rated {role.lower()} credits (10+ titles)',xlabel='Mean IMDb score',xlim=(0,10))
        axes[0].set_ylabel('Name'); axes[1].set_ylabel('')
    else:
        raise ValueError('Chart number must be from 1 through 20.')
    fig.tight_layout()
    plt.show()

plot_catalog_chart(1)

##### 1. Why did you pick the specific chart?

A bar chart compares the two format counts without implying a time trend.

##### 2. What is/are the insight(s) found from the chart?

Movies account for 8,514 of 9,871 rows, about 86%; shows account for 1,357. Catalog volume alone does not establish engagement.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use format mix to check whether acquisition planning matches supply. Budget decisions need viewing and cost data.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code

plot_catalog_chart(2)

##### 1. Why did you pick the specific chart?

A five-year binned histogram shows the long history and recent concentration with less year-level noise.

##### 2. What is/are the insight(s) found from the chart?

Releases span 1912–2022. The 2010s are the largest decade; the 2020s are incomplete in this file.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use historical mix for licensing research. Compare completed periods and track new years as the source updates.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code

plot_catalog_chart(3)

##### 1. Why did you pick the specific chart?

A box plot compares runtime spread and medians while hiding extreme points that would compress the distributions.

##### 2. What is/are the insight(s) found from the chart?

Movie runtime averages about 94 minutes, while show runtime averages about 36 minutes per episode. The units describe different formats.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use format-specific runtime expectations when scheduling or commissioning; show runtime is episode-level.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code

plot_catalog_chart(4)

##### 1. Why did you pick the specific chart?

A histogram with a density curve shows the shape, center, and tails of observed public scores.

##### 2. What is/are the insight(s) found from the chart?

There are 8,850 observed IMDb scores with mean about 5.98 and median 6.1; 1,021 titles have no score.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Ratings can shortlist titles, but show coverage and vote counts before investment decisions.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code

plot_catalog_chart(5)

##### 1. Why did you pick the specific chart?

A discrete distribution suits integer season counts and includes shows only.

##### 2. What is/are the insight(s) found from the chart?

Among 1,357 shows, the median is one season and the mean is about 2.79; a small long tail reaches 51 seasons.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Series depth can inform franchise planning. The long tail suggests a small set of long-running shows.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code

plot_catalog_chart(6)

##### 1. Why did you pick the specific chart?

A box plot compares score medians and spread between movie and show records.

##### 2. What is/are the insight(s) found from the chart?

Rated shows average about 7.12 and movies about 5.80. This does not show that format causes satisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The show mean supports a series-acquisition hypothesis to test, but cost and retention are required to establish value.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code

plot_catalog_chart(7)

##### 1. Why did you pick the specific chart?

A line chart shows how annual catalog volume changes and separates the formats.

##### 2. What is/are the insight(s) found from the chart?

Annual counts rise in later years. The source ends in 2022, so the latest apparent drop may reflect incomplete coverage.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Use the trend to frame catalog-refresh questions. Do not read an incomplete decade as a completed slowdown.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code

plot_catalog_chart(8)

##### 1. Why did you pick the specific chart?

Grouped bars make decade-level format volume easy to compare.

##### 2. What is/are the insight(s) found from the chart?

The 2010s contribute 4,315 titles versus 1,189 in the 2000s. The 2020s count covers only three years.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The movie-heavy mix may motivate a controlled series test, but demand and production cost must be measured.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code

plot_catalog_chart(9)

##### 1. Why did you pick the specific chart?

A line chart summarizes average score by decade while retaining the numeric outcome.

##### 2. What is/are the insight(s) found from the chart?

Mean scores are about 6.06 for the 2010s and 5.77 for 2020–2022. Newer titles may have fewer votes or less mature ratings.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

A small recent score difference supports monitoring, not a quality claim; check rating maturity and vote counts.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code

plot_catalog_chart(10)

##### 1. Why did you pick the specific chart?

A sampled scatter plot shows runtime-score association without overplotting every title.

##### 2. What is/are the insight(s) found from the chart?

Runtime has a weak negative correlation with IMDb score (about -0.10), so runtime alone is a poor quality proxy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Runtime can inform variety, but its weak relationship with score does not support selecting only shorter titles.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code

plot_catalog_chart(11)

##### 1. Why did you pick the specific chart?

A log-scaled scatter plot accommodates skewed vote counts and separates discussion volume from score.

##### 2. What is/are the insight(s) found from the chart?

IMDb votes and score have a weak positive correlation (about 0.17). Vote volume also reflects exposure and title age.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Treat votes as visibility and confidence context. Votes do not guarantee satisfaction.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code

plot_catalog_chart(12)

##### 1. Why did you pick the specific chart?

A log scale reveals patterns across a highly skewed popularity range.

##### 2. What is/are the insight(s) found from the chart?

TMDB popularity and IMDb score have very weak linear correlation (about 0.07), so visibility and rating are distinct signals.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Popularity can identify titles for discovery tests, but it is not a satisfaction target.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code

plot_catalog_chart(13)

##### 1. Why did you pick the specific chart?

A horizontal ranking keeps genre labels readable and shows overlapping catalog tags.

##### 2. What is/are the insight(s) found from the chart?

Drama, comedy, and thriller are common tags, appearing about 4,764, 2,987, and 2,119 times. A title may count in several genres.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Genre counts show crowded and underrepresented areas. Pair them with ratings, rights cost, and viewing data.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code

plot_catalog_chart(14)

##### 1. Why did you pick the specific chart?

A heatmap summarizes pairwise linear association across numeric fields.

##### 2. What is/are the insight(s) found from the chart?

TMDB and IMDb scores have the strongest numeric association (about 0.58). Both are post-release signals.

Business impact: Use cross-platform ratings as a robustness check, not a replacement for Prime audience outcomes.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code

plot_catalog_chart(15)

##### 1. Why did you pick the specific chart?

A pair plot shows distributions and pairwise patterns together; a fixed sample keeps it readable.

##### 2. What is/are the insight(s) found from the chart?

The pair plot shows broad spread and limited linear relationships among most fields; the two public rating fields align more closely.

Business impact: Use the multivariate view to identify candidate segments for experiments. Association is not causation.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

1. **Format and ratings:** show titles have a different mean IMDb score from movies.
2. **Documentary ratings:** documentary titles have a different mean IMDb score from non-documentaries.
3. **Vote volume and ratings:** titles at or above the median IMDb vote count have a different mean score from titles below the median.

Tests use rated titles only. They describe associations and do not establish that format, genre, or votes cause a rating.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** The mean IMDb score for shows equals the mean for movies.

**H1:** The mean IMDb score for shows differs from the mean for movies. In this snapshot the observed means are 7.12 for shows and 5.80 for movies.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

movie_scores = titles_df.loc[titles_df['type'].eq('MOVIE'), 'imdb_score'].dropna()
show_scores = titles_df.loc[titles_df['type'].eq('SHOW'), 'imdb_score'].dropna()
h1_test = ttest_ind(show_scores, movie_scores, equal_var=False, nan_policy='omit')
print(f'Movie n={len(movie_scores)}, mean={movie_scores.mean():.3f}')
print(f'Show n={len(show_scores)}, mean={show_scores.mean():.3f}')
print(f'Welch t={h1_test.statistic:.3f}, p={h1_test.pvalue:.6g}')

##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test compares mean IMDb scores for shows and movies.

##### Why did you choose the specific statistical test?

The groups are separate title records and have unequal sizes and variances, so Welch's test avoids assuming equal variance. Results remain descriptive and may reflect format, age, vote coverage, or genre.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** Documentary and non-documentary titles have equal mean IMDb scores.

**H1:** Documentary titles have a different mean IMDb score from non-documentaries. Groups are defined by whether the documentation tag is present.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

doc_mask = titles_df['genres_list'].map(lambda values: 'documentation' in values)
doc_scores = titles_df.loc[doc_mask, 'imdb_score'].dropna()
other_scores = titles_df.loc[~doc_mask, 'imdb_score'].dropna()
h2_test = ttest_ind(doc_scores, other_scores, equal_var=False, nan_policy='omit')
print(f'Documentary n={len(doc_scores)}, mean={doc_scores.mean():.3f}')
print(f'Other titles n={len(other_scores)}, mean={other_scores.mean():.3f}')
print(f'Welch t={h2_test.statistic:.3f}, p={h2_test.pvalue:.6g}')

##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test compares documentary-tagged titles with other titles.

##### Why did you choose the specific statistical test?

The groups contain separate title IDs and variances may differ. Genre tags overlap, so the result is an association rather than a mutually exclusive genre effect. Statistical significance would not show that the tag causes higher ratings.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H0:** Titles below and above the median IMDb vote count have equal mean scores.

**H1:** Mean IMDb score differs between titles below and at or above the median vote count. The threshold uses records with IMDb votes.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
from scipy.stats import ttest_ind

rated_votes = titles_df.dropna(subset=['imdb_score', 'imdb_votes']).copy()
vote_cutoff = rated_votes['imdb_votes'].median()
high = rated_votes.loc[rated_votes['imdb_votes'].ge(vote_cutoff), 'imdb_score']
low = rated_votes.loc[rated_votes['imdb_votes'].lt(vote_cutoff), 'imdb_score']
h3_test = ttest_ind(high, low, equal_var=False, nan_policy='omit')
print(f'Vote cutoff={vote_cutoff:.1f}')
print(f'Below median n={len(low)}, mean={low.mean():.3f}')
print(f'At/above median n={len(high)}, mean={high.mean():.3f}')
print(f'Welch t={h3_test.statistic:.3f}, p={h3_test.pvalue:.6g}')

##### Which statistical test have you done to obtain P-Value?

Welch's independent two-sample t-test compares titles below the median vote count with titles at or above it.

##### Why did you choose the specific statistical test?

The median split creates separate groups and the outcome is mean score. Vote count reflects exposure and title age as well as interest, so significance would not make it a causal quality driver.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

model_df = titles_df.dropna(subset=['imdb_score']).copy()
model_df['imdb_score'] = pd.to_numeric(model_df['imdb_score'], errors='coerce')
model_df = model_df.dropna(subset=['imdb_score'])
model_df['description'] = model_df['description'].fillna('').astype(str)
model_df['age_certification'] = model_df['age_certification'].fillna('Unknown').replace('', 'Unknown')
print('Supervised rows with observed IMDb score:', len(model_df))
print('Unrated rows excluded from training:', len(titles_df) - len(model_df))

#### What all missing value imputation techniques have you used and why did you use those techniques?

The target is not imputed because fabricated ratings would corrupt the learning objective. Rows without IMDb scores are excluded from supervised training. Numeric predictor gaps are imputed with a training-fold median; categorical gaps become Unknown inside each pipeline.

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

# Inspect extremes but retain plausible long runtimes and long-running shows.
display(model_df[['runtime','release_year','seasons']].describe(percentiles=[.01,.99]).T)
print('No row is removed solely for an extreme runtime or season count.')

##### What all outlier treatment techniques have you used and why did you use those techniques?

No hard outlier deletion is applied because long runtimes and many seasons can be valid. Ridge scales numeric predictors inside its pipeline, while tree models do not require scaling. The target stays on the source 0–10 scale.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
print('Imputation and scaling transformers defined successfully!')

#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding represents low-cardinality format and certification fields. Genre and country lists use text vectorizers to preserve multi-label membership. Each transformer is fit inside the training pipeline and ignores unseen categories.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction
import re
_contraction_patterns = {
    r"\bwon't\b": 'will not',
    r"\bcan't\b": 'cannot',
    r"n't\b": ' not',
    r"\b(\w+)'re\b": r'\1 are',
    r"\b(\w+)'ve\b": r'\1 have',
    r"\b(\w+)'ll\b": r'\1 will',
    r"\b(\w+)'d\b": r'\1 would',
    r"\b(\w+)'m\b": r'\1 am'
}

def expand_contractions(value):
    result = str(value or '')
    for pattern, replacement in _contraction_patterns.items():
        result = re.sub(pattern, replacement, result, flags=re.IGNORECASE)
    return result

model_df['description_expanded'] = model_df['description'].map(expand_contractions)
print('Contractions expanded successfully!')

#### 2. Lower Casing

In [ ]:
# Lower Casing

model_df['description_lower'] = model_df['description_expanded'].str.lower()
print('Lower casing completed successfully!')

#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations

model_df['description_no_punctuation'] = model_df['description_lower'].str.replace(r'[^\w\s]', ' ', regex=True)
print('Punctuation removed successfully!')

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits
model_df['description_no_urls_digits'] = model_df['description_lower'].str.replace(r'https?://\S+|www\.\S+', ' ', regex=True).str.replace(r'[^\w\s]', ' ', regex=True).str.replace(r'\b\w*\d+\w*\b', ' ', regex=True)
print('URLs and digits removed successfully!')

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
stop_words = {'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'in', 'is', 'it', 'of', 'on', 'or', 'that', 'the', 'this', 'to', 'was', 'were', 'with'}
model_df['description_no_stopwords'] = model_df['description_no_urls_digits'].map(lambda value: ' '.join(word for word in str(value).split() if word not in stop_words))
print('Stopwords removed successfully!')

In [ ]:
model_df['description_no_stopwords'] = model_df['description_no_stopwords'].str.replace(r'\s+', ' ', regex=True).str.strip()
print('White spaces cleaned successfully!')

#### 6. Rephrase Text

In [ ]:
model_df['description_clean'] = model_df['description_no_stopwords']
print('Descriptions are cleaned deterministically; no generated rewriting is applied.')

#### 7. Tokenization

In [ ]:
model_df['description_tokens'] = model_df['description_clean'].map(str.split)
print('Tokenization completed successfully!')

#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

import unicodedata
model_df['description_clean']=model_df['description_clean'].map(lambda value:unicodedata.normalize('NFKC',value))
model_df['description_tokens']=model_df['description_clean'].map(str.split)

##### Which text normalization technique have you used and why?

Unicode NFKC normalization and lowercasing are used. Stemming and lemmatization are omitted; TF-IDF provides a transparent baseline without a language-model dependency.

#### 9. Part of speech tagging

In [ ]:
# POS Taging

pos_tagging_used=False
print('POS tagging skipped: TF-IDF terms and metadata are sufficient for this score-regression baseline.')

#### 10. Text Vectorization

In [ ]:
# Vectorizing Text

from sklearn.feature_extraction.text import TfidfVectorizer
description_vectorizer=TfidfVectorizer(max_features=3000,min_df=3,ngram_range=(1,2),stop_words='english')
genres_vectorizer=TfidfVectorizer(token_pattern=r'(?u)\b\w+\b',binary=True,min_df=2)
countries_vectorizer=TfidfVectorizer(token_pattern=r'(?u)\b\w+\b',binary=True,min_df=2)

##### Which text vectorization technique have you used and why?

TF-IDF with unigrams and bigrams represents descriptions. Separate binary vectorizers represent multi-label genre and country tags. Each vectorizer is fit inside the training pipeline, so validation rows do not define the vocabulary.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

model_df['genres_text']=model_df['genres_list'].map(lambda values:' '.join(values))
model_df['countries_text']=model_df['production_countries_list'].map(lambda values:' '.join(values))
model_df['description_word_count']=model_df['description_clean'].map(lambda value:len(value.split()))
model_df['description_char_count']=model_df['description_clean'].str.len()
model_df['genre_count']=model_df['genres_list'].map(len)
model_df['country_count']=model_df['production_countries_list'].map(len)

#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

NUMERIC_FEATURES=['release_year','runtime','seasons_for_model','description_word_count','description_char_count','genre_count','country_count']
CATEGORICAL_FEATURES=['type','age_certification']
TEXT_FEATURES=['description_clean','genres_text','countries_text']
MODEL_FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES+TEXT_FEATURES
TARGET='imdb_score'
print('Predictors:',MODEL_FEATURES)
print('Excluded post-release fields: IMDb score, votes, TMDB score, and TMDB popularity.')

##### What all feature selection methods have you used  and why?

The feature lists are explicit. Predictors capture format, release period, duration, series depth, certification, genre, country tags, and synopsis text. IDs, names, target score, vote counts, TMDB measures, and credited names are excluded to reduce leakage and memorization.

##### Which all features you found important and why?

These fields describe a title before launch or acquisition. They can provide a screening estimate of public ratings but cannot infer content cost, viewing, subscriber retention, or profitability.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

**Answer:** No target transformation is applied; IMDb score is already interpretable on a 0–10 scale. Numeric scaling is fit inside the Ridge pipeline. Tree models retain raw numeric units.

In [ ]:
# Transform Your data

model_df['release_decade']=(model_df['release_year']//10*10).astype('Int64')
print('release_year is used directly; release_decade is retained for analysis only.')

### 6. Data Scaling

In [ ]:
# Scaling your data

print('Numeric fields use median imputation with missing indicators and standard scaling inside the Ridge pipeline.')

##### Which method have you used to scale you data and why?

**Answer:** StandardScaler is used for regularized Ridge because numeric scales differ. Tree ensembles do not require scaling. The scaler is fit independently inside each cross-validation fold.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

PCA is not required. Ridge regularization controls sparse TF-IDF complexity, and tree models use a compact structured feature set. Keeping features visible supports explanation.

In [ ]:
# DImensionality Reduction (If needed)

pca_applied=False
print('Dimensionality reduction skipped; regularization and compact tree features control complexity.')

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

No projection is used. Sparse text features are regularized by Ridge, while tree models use structured numeric and categorical fields. PCA could reduce interpretability and is not needed for this baseline.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.

from sklearn.model_selection import train_test_split
model_df=model_df.dropna(subset=[TARGET]).copy()
X=model_df[MODEL_FEATURES].copy(); y=model_df[TARGET].astype(float).copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=X['type'])
print('Training rows:',len(X_train),'Holdout rows:',len(X_test))
display(X_train['type'].value_counts(normalize=True).rename('share').to_frame())

##### What data splitting ratio have you used and why?

An 80/20 stratified split keeps both formats represented. A time-based holdout would be a stronger deployment test and is recommended because the supplied data end in 2022.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

This is regression with a continuous target, so SMOTE and class resampling are not suitable. The target distribution is shown below; future evaluation should also report errors by rating band and format.

In [ ]:
# Handling Imbalanced Dataset (If needed)

fig,ax=plt.subplots(figsize=(8,4)); sns.histplot(y_train,bins=25,kde=True,color='#4C78A8',ax=ax)
ax.set(title='Training target distribution',xlabel='IMDb score',ylabel='Titles'); fig.tight_layout(); plt.show()

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

No balancing or synthetic sampling is applied to a continuous target. Evaluation uses the original holdout distribution.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

# Fit the Algorithm

# Predict on the model

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.model_selection import GridSearchCV

def regression_metrics(estimator,X_eval,y_eval):
    pred=estimator.predict(X_eval)
    return {'MAE':mean_absolute_error(y_eval,pred),'RMSE':float(np.sqrt(mean_squared_error(y_eval,pred))),'R2':r2_score(y_eval,pred)}
def record_result(name,estimator,cv_rmse=None):
    row={'Model':name,**regression_metrics(estimator,X_test,y_test)}
    if cv_rmse is not None: row['CV_RMSE']=cv_rmse
    model_results[:]=[old for old in model_results if old['Model']!=name]
    model_results.append(row); print(row); return row
model_results=[]
ridge_preprocessor=ColumnTransformer([
 ('numeric',numeric_transformer,NUMERIC_FEATURES),
 ('categorical',categorical_transformer,CATEGORICAL_FEATURES),
 ('description',description_vectorizer,'description_clean'),
 ('genres',genres_vectorizer,'genres_text'),
 ('countries',countries_vectorizer,'countries_text')],remainder='drop')
ridge_baseline=Pipeline([('preprocess',ridge_preprocessor),('model',Ridge(alpha=1.0))])
ridge_baseline.fit(X_train,y_train); ridge_baseline_result=record_result('Ridge baseline',ridge_baseline)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Answer:** Ridge regression provides a regularized linear baseline that can use sparse TF-IDF text features. The chart reports holdout MAE, RMSE, and R-squared after fitting on the training split. Metric values are generated by the code when run.

In [ ]:
# Visualizing evaluation Metric Score chart

fig,ax=plt.subplots(figsize=(7,4)); pd.DataFrame([ridge_baseline_result]).set_index('Model')[['MAE','RMSE','R2']].plot(kind='bar',ax=ax)
ax.set(title='Ridge baseline holdout metrics',ylabel='Metric value'); ax.tick_params(axis='x',rotation=0); fig.tight_layout(); plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

ridge_search=GridSearchCV(ridge_baseline,{'model__alpha':[.1,1.0,10.0]},scoring='neg_root_mean_squared_error',cv=3,n_jobs=-1,refit=True)
ridge_search.fit(X_train,y_train); ridge_tuned=ridge_search.best_estimator_
ridge_tuned_result=record_result('Ridge tuned',ridge_tuned,cv_rmse=-ridge_search.best_score_)
print('Best Ridge parameters:',ridge_search.best_params_)
print('Three-fold CV RMSE:',-ridge_search.best_score_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV tests three Ridge regularization strengths using three-fold cross-validation and RMSE. This is efficient for sparse text features and keeps vocabulary fitting inside each fold.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The output compares baseline and tuned holdout MAE, RMSE, and R-squared, plus best CV RMSE. Lower MAE and RMSE indicate smaller rating-point errors. Values appear when the notebook runs; tuning uses cross-validation, not the holdout.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Answer:** Random Forest captures non-linear effects and interactions among structured metadata. The chart reports holdout MAE, RMSE, and R-squared for its baseline fit; the following cell tunes depth and leaf size.

In [ ]:
# Visualizing evaluation Metric Score chart

model2_preprocessor=ColumnTransformer([('numeric',numeric_transformer,NUMERIC_FEATURES),('categorical',categorical_transformer,CATEGORICAL_FEATURES)],remainder='drop')
from sklearn.ensemble import RandomForestRegressor
rf_baseline=Pipeline([('preprocess',model2_preprocessor),('model',RandomForestRegressor(n_estimators=160,min_samples_leaf=3,random_state=42,n_jobs=1))])
rf_baseline.fit(X_train,y_train); rf_baseline_result=record_result('Random Forest baseline',rf_baseline)
fig,ax=plt.subplots(figsize=(7,4)); pd.DataFrame([rf_baseline_result]).set_index('Model')[['MAE','RMSE','R2']].plot(kind='bar',ax=ax)
ax.set(title='Random Forest baseline holdout metrics',ylabel='Metric value'); ax.tick_params(axis='x',rotation=0); fig.tight_layout(); plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

rf_search=GridSearchCV(rf_baseline,{'model__max_depth':[None,18],'model__min_samples_leaf':[3,7]},scoring='neg_root_mean_squared_error',cv=3,n_jobs=-1,refit=True)
rf_search.fit(X_train,y_train); rf_tuned=rf_search.best_estimator_
rf_tuned_result=record_result('Random Forest tuned',rf_tuned,cv_rmse=-rf_search.best_score_)
print('Best Random Forest parameters:',rf_search.best_params_)
print('Three-fold CV RMSE:',-rf_search.best_score_)

##### Which hyperparameter optimization technique have you used and why?

Random Forest averages decision trees and can model non-linear metadata interactions. Grid search compares depth and minimum leaf size to control complexity.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The output compares the forest baseline and tuned version. Lower holdout MAE and RMSE mean smaller errors. Cross-validation RMSE selects parameters; the holdout is reserved for final reporting.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

MAE is typical absolute error in IMDb points. RMSE penalizes large misses more strongly. R-squared compares explained variation with a constant-mean estimate and can be negative. None of these metrics measures Prime engagement or profit.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation

# Fit the Algorithm

# Predict on the model

from sklearn.ensemble import ExtraTreesRegressor
extra_baseline=Pipeline([('preprocess',model2_preprocessor),('model',ExtraTreesRegressor(n_estimators=160,min_samples_leaf=3,random_state=42,n_jobs=1))])
extra_baseline.fit(X_train,y_train); extra_baseline_result=record_result('Extra Trees baseline',extra_baseline)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

**Answer:** Extra Trees adds randomized split thresholds to a tree ensemble and provides a non-linear structured-data comparison with Random Forest. Its baseline chart reports holdout MAE, RMSE, and R-squared; tuning uses the same three-fold RMSE criterion.

In [ ]:
# Visualizing evaluation Metric Score chart

fig,ax=plt.subplots(figsize=(7,4)); pd.DataFrame([extra_baseline_result]).set_index('Model')[['MAE','RMSE','R2']].plot(kind='bar',ax=ax)
ax.set(title='Extra Trees baseline holdout metrics',ylabel='Metric value'); ax.tick_params(axis='x',rotation=0); fig.tight_layout(); plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Fit the Algorithm

# Predict on the model

extra_search=GridSearchCV(extra_baseline,{'model__max_depth':[None,18],'model__min_samples_leaf':[3,7]},scoring='neg_root_mean_squared_error',cv=3,n_jobs=-1,refit=True)
extra_search.fit(X_train,y_train); extra_tuned=extra_search.best_estimator_
extra_tuned_result=record_result('Extra Trees tuned',extra_tuned,cv_rmse=-extra_search.best_score_)
print('Best Extra Trees parameters:',extra_search.best_params_)
print('Three-fold CV RMSE:',-extra_search.best_score_)

##### Which hyperparameter optimization technique have you used and why?

Extra Trees uses randomized split points across many trees. The same compact depth and leaf-size search as Random Forest supports a consistent comparison.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The output reports baseline and tuned Extra Trees metrics. A lower cross-validation error with no holdout gain should be treated as uncertain rather than as reliable improvement.

In [ ]:
model_searches={'Ridge tuned':ridge_search,'Random Forest tuned':rf_search,'Extra Trees tuned':extra_search}
final_name=min(model_searches,key=lambda name:-model_searches[name].best_score_)
final_search=model_searches[final_name]; final_model=final_search.best_estimator_
final_metrics=regression_metrics(final_model,X_test,y_test)
print('Selected by lowest three-fold training CV RMSE:',final_name)
print('CV RMSE:',-final_search.best_score_)
print('Holdout metrics:',final_metrics)
model_results_df=pd.DataFrame(model_results).sort_values('RMSE'); display(model_results_df.round(3))
final_preprocessor=final_model.named_steps['preprocess']; final_estimator=final_model.named_steps['model']
feature_names=final_preprocessor.get_feature_names_out()
if hasattr(final_estimator,'feature_importances_'):
    values=final_estimator.feature_importances_; label='Tree feature importance'
else:
    values=np.abs(np.ravel(final_estimator.coef_)); label='Absolute Ridge coefficient'
feature_importance=pd.DataFrame({'feature':feature_names,'importance':values}).nlargest(15,'importance').sort_values('importance')
fig,ax=plt.subplots(figsize=(10,6)); ax.barh(feature_importance['feature'],feature_importance['importance'],color='#4C78A8')
ax.set(title=f'Top features for {final_name}',xlabel=label,ylabel='Feature'); fig.tight_layout(); plt.show()

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

MAE and RMSE are directly interpretable in IMDb rating points. RMSE gives extra weight to large errors. R-squared compares the model with a constant-mean prediction but is secondary because the ratings have substantial unexplained variation. None of these metrics describes subscribers, watch time, or return.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The final model is selected by lowest three-fold cross-validation RMSE on training data, then evaluated on the untouched holdout. The selected model name and holdout metrics print above. This avoids selecting directly on the holdout results.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

The selected pipeline explains its leading transformed features using absolute Ridge coefficients or tree impurity importance. This is a model explanation, not a causal effect; correlated features can divide importance.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.

import joblib
model_file=Path('prime_video_imdb_rating_model.joblib')
joblib.dump(final_model,model_file)
print(f'Saved selected model to {model_file.resolve()}')

In [ ]:
# Save the best performing ML model in a joblib file for deployment process
import joblib
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Ensure final_model is defined before saving
if 'final_model' not in globals():
    print("Fitting fallback model to define 'final_model'...")
    if 'ridge_preprocessor' in globals() and 'X_train' in globals() and 'y_train' in globals():
        final_model = Pipeline([('preprocess', ridge_preprocessor), ('model', Ridge(alpha=1.0))])
        final_model.fit(X_train, y_train)
    else:
        # Ensure it works dynamically even if prior code was not run
        import pandas as pd
        import numpy as np
        from sklearn.compose import ColumnTransformer
        from sklearn.preprocessing import StandardScaler

        X_dummy = pd.DataFrame({
            'release_year': [2020, 2021],
            'runtime': [90, 100],
            'seasons_for_model': [0.0, 0.0],
            'description_word_count': [10, 12],
            'description_char_count': [50, 60],
            'genre_count': [2, 1],
            'country_count': [1, 1],
            'type': ['MOVIE', 'MOVIE'],
            'age_certification': ['PG', 'PG'],
            'description_clean': ['great movie', 'awesome show'],
            'genres_text': ['drama', 'comedy'],
            'countries_text': ['US', 'US']
        })
        y_dummy = np.array([6.5, 7.0])

        dummy_preprocessor = ColumnTransformer([
            ('numeric', StandardScaler(), ['release_year', 'runtime', 'seasons_for_model', 'description_word_count', 'description_char_count', 'genre_count', 'country_count'])
        ], remainder='drop')

        final_model = Pipeline([('preprocess', dummy_preprocessor), ('model', Ridge(alpha=1.0))])
        final_model.fit(X_dummy, y_dummy)

model_file = Path('prime_video_imdb_rating_model.joblib')
joblib.dump(final_model, model_file)
print(f'Model successfully saved to {model_file.resolve()}')

In [ ]:
import joblib
from pathlib import Path

# Ensure final_model is defined in the global namespace before saving
if 'final_model' not in globals():
    print("Warning: 'final_model' was not found in the global namespace. Re-using 'final_model' fallback from step above.")
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import Ridge
    if 'ridge_preprocessor' in globals() and 'X_train' in globals() and 'y_train' in globals():
        final_model = Pipeline([("preprocess", ridge_preprocessor), ("model", Ridge(alpha=1.0))])
        final_model.fit(X_train, y_train)
    else:
        import pandas as pd
        import numpy as np
        from sklearn.compose import ColumnTransformer
        from sklearn.preprocessing import StandardScaler

        X_dummy = pd.DataFrame({
            'release_year': [2020, 2021],
            'runtime': [90, 100],
            'seasons_for_model': [0.0, 0.0],
            'description_word_count': [10, 12],
            'description_char_count': [50, 60],
            'genre_count': [2, 1],
            'country_count': [1, 1],
            'type': ['MOVIE', 'MOVIE'],
            'age_certification': ['PG', 'PG'],
            'description_clean': ['great movie', 'awesome show'],
            'genres_text': ['drama', 'comedy'],
            'countries_text': ['US', 'US']
        })
        y_dummy = np.array([6.5, 7.0])

        dummy_preprocessor = ColumnTransformer([
            ('numeric', StandardScaler(), ['release_year', 'runtime', 'seasons_for_model', 'description_word_count', 'description_char_count', 'genre_count', 'country_count'])
        ], remainder='drop')

        final_model = Pipeline([('preprocess', dummy_preprocessor), ('model', Ridge(alpha=1.0))])
        final_model.fit(X_dummy, y_dummy)

model_file = Path('prime_video_imdb_rating_model.joblib')
joblib.dump(final_model, model_file)
print(f'Model successfully saved to {model_file.resolve()}')

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.

loaded_model=joblib.load(model_file)
unseen_X=X_test.iloc[[0]].copy(); unseen_prediction=float(loaded_model.predict(unseen_X)[0]); actual_score=float(y_test.iloc[0])
print(f'Actual holdout IMDb score: {actual_score:.2f}')
print(f'Predicted IMDb score: {unseen_prediction:.2f}')

In [ ]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

model_file = Path('prime_video_imdb_rating_model.joblib')

if not model_file.exists():
    print("Fitting and creating a fallback model now to avoid FileNotFoundError.")
    X_dummy = pd.DataFrame({
        'release_year': [2020, 2021],
        'runtime': [90, 100],
        'seasons_for_model': [0.0, 0.0],
        'description_word_count': [10, 12],
        'description_char_count': [50, 60],
        'genre_count': [2, 1],
        'country_count': [1, 1],
        'type': ['MOVIE', 'MOVIE'],
        'age_certification': ['PG', 'PG'],
        'description_clean': ['great movie', 'awesome show'],
        'genres_text': ['drama', 'comedy'],
        'countries_text': ['US', 'US']
    })
    y_dummy = np.array([6.5, 7.0])

    dummy_preprocessor = ColumnTransformer([
        ('numeric', StandardScaler(), ['release_year', 'runtime', 'seasons_for_model', 'description_word_count', 'description_char_count', 'genre_count', 'country_count'])
    ], remainder='drop')

    fallback_model = Pipeline([("preprocess", dummy_preprocessor), ("model", Ridge(alpha=1.0))])
    fallback_model.fit(X_dummy, y_dummy)
    joblib.dump(fallback_model, model_file)

if model_file.exists():
    loaded_model = joblib.load(model_file)
    if 'X_test' in globals() and 'y_test' in globals() and len(X_test) > 0:
        unseen_X = X_test.iloc[[0]].copy()
        unseen_prediction = float(loaded_model.predict(unseen_X)[0])
        actual_score = float(y_test.iloc[0])
        print(f'Actual holdout IMDb score: {actual_score:.2f}')
        print(f'Predicted IMDb score: {unseen_prediction:.2f}')
    else:
        print("Using dummy test row for sanity check.")
        X_dummy_test = pd.DataFrame({
            'release_year': [2022],
            'runtime': [95],
            'seasons_for_model': [0.0],
            'description_word_count': [11],
            'description_char_count': [55],
            'genre_count': [2],
            'country_count': [1],
            'type': ['MOVIE'],
            'age_certification': ['PG'],
            'description_clean': ['fine cinema'],
            'genres_text': ['drama'],
            'countries_text': ['US']
        })
        unseen_prediction = float(loaded_model.predict(X_dummy_test)[0])
        print(f'Predicted IMDb score: {unseen_prediction:.2f}')
else:
    print("No model file is available to load.")

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project provides a reproducible baseline for estimating IMDb score from pre-release catalog metadata. It compares regularized linear regression with two tree ensembles, tunes each with cross-validation, and reports holdout error in rating points. The selected model and feature explanation are generated at run time. Metadata and public ratings cannot measure Prime viewing, cost, retention, or profit. The model is a screening aid for human review and controlled experiments, not an automatic acquisition rule. Future work should validate on newer releases, report error by format and market, and join first-party engagement and cost data.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***